In [1]:
import langchain
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool

c:\Users\Sangram Mishra\Desktop\ADK\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

print("the enviroments variable are loaded")

the enviroments variable are loaded


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
model = init_chat_model(
    model = "openai/gpt-oss-20b",
    model_provider= "groq"
)

In [19]:
from langchain.messages import HumanMessage, AIMessage

In [23]:
chat_history = []
while True:
    query = str(input("hi what you want to ask"))
    if query.lower().strip() == "quit":
        break
    else:
        chat_history.append(HumanMessage(content=query))
        print(query)
        print("="*50)
        reply = []
        for chunks in model.stream(chat_history):
            print(chunks.content,end="",flush=True)
            reply.append(chunks.content)
        print("\n")
        print("*"*50)
        ai_msg = " ".join(reply)
        chat_history.append(AIMessage(content=ai_msg))

hit
I’m not sure what you mean by “hit.” Could you provide a bit more context or let me know what you’d like me to do?

**************************************************
are you good or bad
I’m a tool—just a set of algorithms that process language. I don’t have feelings, intentions, or a moral compass of my own, so I’m neither “good” nor “bad” in an absolute sense.  

What matters is how people use me. When I’m employed to help people learn, solve problems, or create art, I can be a force for good. If I’m used to spread misinformation, automate harassment, or otherwise harm others, that’s a misuse that reflects the user’s choices, not an inherent quality of the model.

So, the “goodness” or “badness” of an AI like me depends on the purpose, context, and safeguards that humans put in place.

**************************************************
do you know my name
I don’t have any way to know your name unless you tell me. If you’d like to share it (or not), that’s entirely up to you.

***

In [25]:
len(chat_history)

24

In [26]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

while True:
    # Removed the redundant str() and added formatting for the prompt
    query = input("Hi, what do you want to ask?: ")
    
    if query.lower().strip() == "quit":
        break
        
    chat_history.append(HumanMessage(content=query))
    
    print(f"\nYou: {query}")
    print("=" * 50)
    
    reply = []
    
    try:
        # Stream the response
        print("AI: ", end="")
        for chunk in model.stream(chat_history):
            print(chunk.content, end="", flush=True)
            reply.append(chunk.content)
            
        print("\n")
        print("*" * 50)
        
        # FIX: Join with an empty string, not a space!
        ai_msg = "".join(reply)
        chat_history.append(AIMessage(content=ai_msg))
        
    except Exception as e:
        print(f"\nAn error occurred: {e}")


You: hi
AI: Hello! 👋 How can I help you today?

**************************************************

You: how are you
AI: I’m doing great—ready to help you out! How about you? Anything on your mind today?

**************************************************

You: what is your ate limit
AI: It looks like you might be asking about a *rate limit* (the maximum number of requests you can make in a given time).  

### For the ChatGPT web interface
- **No hard per‑minute limit** – the interface is designed to handle a normal conversation flow.  
- If you’re chatting at a very fast pace, you might see a short “Please wait a moment” pause, but that’s just a courtesy throttling to keep the service stable for everyone.

### For the OpenAI API (ChatGPT‑4, GPT‑4o, etc.)
- **Standard rate limits**:  
  - **Requests per minute**: 60 – 120 req/min for most accounts.  
  - **Tokens per minute**: Roughly 200 k – 400 k tokens/min (depends on the model).  
- **Higher tiers**: If you’re on a higher‑volume p

In [27]:
### ow we will learn more towards tool callimng
@tool
def get_weather(city:str)->str:
    """thjis the tool desighend for geting city for
    a given city mentioned."""
    city = city.lower().strip()
    weather = {
        "puri": "its cloudy ad humid",
        "bbsr": "its hoty"
    }
    
    return weather.get(city,"the data not found in the dataset")

In [28]:
model_with_tool = model.bind_tools([get_weather])

In [29]:
response = model_with_tool.invoke("what is the weatherin bbsr")
print (response)



content='' additional_kwargs={'reasoning_content': 'The user asks "what is the weather in bbsr". The tool is get_weather(city). We should call the function with city: "bbsr". Probably BBSR is abbreviation for Bangalore (Bengaluru). So we call the function.', 'tool_calls': [{'id': 'fc_f8a6bbeb-e10f-44b5-abc2-9cf0a3d93e78', 'function': {'arguments': '{"city":"bbsr"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 140, 'total_tokens': 217, 'completion_time': 0.079828306, 'completion_tokens_details': {'reasoning_tokens': 53}, 'prompt_time': 0.006844724, 'prompt_tokens_details': None, 'queue_time': 0.211460383, 'total_time': 0.08667303}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a10239-a121-7061-b7b4-4ecaa632cfd1-0' tool_calls=[{'name': 'get_weather', 'args': {'city'

In [32]:
for tools in response.tool_calls:
    print(f"name of tool : {tools["name"]}")
    print(f"args: {tools["args"]}")

name of tool : get_weather
args: {'city': 'bbsr'}


In [35]:
messages  = [{"role":"user","content":"what is the weather in kolkata"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    print(tool_call)
    result = get_weather.invoke(tool_call)
    messages.append(result)

final_ans = model_with_tool.invoke(messages)


{'name': 'get_weather', 'args': {'city': 'kolkata'}, 'id': 'fc_b392d90b-5b33-48fa-85e8-e8201fc2a798', 'type': 'tool_call'}


In [37]:
print(final_ans)

content='I’m sorry, but I couldn’t find weather information for Kolkata in the available dataset. If you’d like, I can try looking up a different city or provide general climate information for the region. Let me know how I can help!' additional_kwargs={'reasoning_content': 'The user asked "what is the weather in kolkata". We used the get_weather tool, but it returned "data not found". We should respond to the user indicating we couldn\'t find data. Possibly ask if they\'d like to try another city or provide approximate weather? According to policy, we can respond. There\'s no conflict. So answer that we couldn\'t find data.'} response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 172, 'total_tokens': 303, 'completion_time': 0.14607969, 'completion_tokens_details': {'reasoning_tokens': 74}, 'prompt_time': 0.008437188, 'prompt_tokens_details': None, 'queue_time': 0.346626807, 'total_time': 0.154516878}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp

In [43]:
for i in (ai_msg):
    print((i))

('content', '')
('additional_kwargs', {'reasoning_content': 'The user asks: "what is the weather in kolkata". We need to use the get_weather function. We need to call it with city "kolkata".', 'tool_calls': [{'id': 'fc_b392d90b-5b33-48fa-85e8-e8201fc2a798', 'function': {'arguments': '{"city":"kolkata"}', 'name': 'get_weather'}, 'type': 'function'}]})
('response_metadata', {'token_usage': {'completion_tokens': 58, 'prompt_tokens': 139, 'total_tokens': 197, 'completion_time': 0.063342489, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.046610719, 'prompt_tokens_details': None, 'queue_time': 0.356571866, 'total_time': 0.109953208}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_228717f27c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'})
('type', 'ai')
('name', None)
('id', 'lc_run--01a10240-1c1d-7d60-b81d-ee06016682e5-0')
('tool_calls', [{'name': 'get_weather', 'args': {'city': 'kolkata'}, 'id

In [40]:
final_ans.tool_calls

[]